In [1]:
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\manju\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\manju\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
#%pip install bert-score rouge-score nltk

In [3]:
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
import pandas as pd
# 1. LOAD DATA
df = pd.read_csv("ai_teacher_results.csv")

# 2. PREPARE REFERENCE AND GENERATED ANSWERS
references = (
    df["Teacher Response"]
    .fillna("")
    .astype(str)
    .tolist()
)
candidates = (
    df["GeneratedAnswer"]
    .fillna("")
    .astype(str)
    .tolist()
)

# 3. BERTSCORE
P, R, F1 = bert_score(
    candidates,
    references,
    lang="en",
    verbose=True
)
df["BERT"] = F1.cpu().numpy()

# 4. ROUGE
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"],use_stemmer=True)
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for candidate, reference in zip(candidates, references):
    scores = rouge.score(reference, candidate)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

df["ROUGE-1"] = rouge1_scores
df["ROUGE-2"] = rouge2_scores
df["ROUGE-L"] = rougeL_scores

# 5. METEOR
meteor_scores = []

for candidate, reference in zip(candidates, references):
    candidate_tokens = candidate.split()
    reference_tokens = reference.split()
    score = meteor_score([reference_tokens], candidate_tokens)
    meteor_scores.append(score)

df["METEOR"] = meteor_scores

# 6. ROUND SCORES

similarity_columns = [
    "BERT",
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L",
    "METEOR"
]
df[similarity_columns] = (df[similarity_columns].round(3))

# 7. QUESTION-WISE SIMILARITY DATAFRAME

similarity_result = df[
    [
        "User Query",
        "BERT",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "METEOR"
    ]
].copy()

# DISPLAY QUESTION-WISE DATAFRAME

display(similarity_result)

# 8. CALCULATE AVERAGES

average_bert = df["BERT"].mean()
average_rouge1 = df["ROUGE-1"].mean()
average_rouge2 = df["ROUGE-2"].mean()
average_rougeL = df["ROUGE-L"].mean()
average_meteor = df["METEOR"].mean()

# 9. CREATE AVERAGE SIMILARITY DATAFRAME

average_similarity = pd.DataFrame({
    "Metric": ["BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"],
    "Average Score": [average_bert, average_rouge1, average_rouge2, average_rougeL, average_meteor]
})

# ROUND AVERAGE SCORES
average_similarity["Average Score"] = (average_similarity["Average Score"].round(3))

# DISPLAY AVERAGE DATAFRAME
display(average_similarity)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 11.74 seconds, 1.70 sentences/sec


,User Query,BERT,ROUGE-1,ROUGE-2,ROUGE-L,METEOR
0,What is AI?,0.906,0.375,0.065,0.312,0.152
1,How does AI differ from traditional programming?,0.910,0.393,0.136,0.262,0.358
2,What is Machine Learning?,0.911,0.472,0.200,0.333,0.367
3,What is Deep Learning?,0.924,0.462,0.317,0.431,0.470
4,"What is the difference between AI, ML, and Dee...",0.929,0.667,0.395,0.590,0.558
5,What is training data?,0.915,0.464,0.296,0.429,0.221
6,What is a Machine Learning model?,0.921,0.418,0.215,0.358,0.426
7,What is supervised learning?,0.867,0.107,0.000,0.071,0.081
8,What is unsupervised learning?,0.911,0.250,0.065,0.219,0.162
9,What is reinforcement learning?,0.894,0.222,0.033,0.190,0.117


,Metric,Average Score
0,BERT,0.899
1,ROUGE-1,0.330
2,ROUGE-2,0.130
3,ROUGE-L,0.277
4,METEOR,0.257


In [4]:
df.to_csv(
    "ai_teacher_evaluation_Similarity_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

In [5]:
average_similarity.to_csv("ai_teacher_evaluation_Similarity_Average.csv", index=False,
    encoding="utf-8-sig")